In [0]:
# Charger la table
df = spark.table("workspace.default.imdb_tmdb_movie_metadata_big_dataset_1_m")

# Afficher le schéma et quelques lignes
df.printSchema()
display(df.limit(5))

In [0]:
# Statistiques descriptives de base
display(df.describe())

#Data Quality Checks 
Vérifiez la qualité de vos données (valeurs nulles, doublons, etc.).

In [0]:
# Vérifier les valeurs nulles par colonne
from pyspark.sql.functions import col, count, when
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
display(null_counts)

In [0]:
# Rechercher des doublons sur un identifiant unique (ex: 'id' ou 'imdb_id')
# Remplacez 'id' par la clé primaire réelle de votre table
if 'id' in df.columns:
    dup_counts = df.groupBy("id").count().filter("count > 1")
    display(dup_counts)

# Transformations et Analyse
transformations et des analyses métier.


In [0]:
#  Nettoyage et sélection de colonnes pertinentes
# Exemple : Garder les films sortis après 2000 et avec un budget non nul
df_clean = df.filter((col("release_year") > 2000) & (col("budget").isNotNull()))



In [0]:
# Agrégation - Nombre de films par genre
from pyspark.sql.functions import explode, split, count, regexp_replace
# La colonne réelle est 'genres_list' au format "['Drama', 'Thriller']" : on retire crochets et guillemets avant de séparer sur ', '
df_genres = df_clean.withColumn("genre", explode(split(regexp_replace(col("genres_list"), "[\\[\\]']", ""), ", ")))
display(df_genres.groupBy("genre").agg(count("*").alias("nombre_films")).orderBy("nombre_films", ascending=False))


In [0]:

# Sauvegarder le résultat dans une nouvelle table Delta (couche "Silver")
df_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.default.films_analyses")